In [1]:
# Function 7 - BBO Capstone Project
# Strategy: Scaled Gaussian Process + EI/UCB Hybrid
# Function 7 is a 6D black-box optimization problem.

import numpy as np
from scipy.stats import norm, qmc
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel


# --------------------------------------------------
# 1. Function 7 data
# --------------------------------------------------

X = np.array([
    [0.27262382, 0.32449536, 0.89710881, 0.83295115, 0.15406269, 0.79586362],
    [0.54300258, 0.92469390, 0.34156746, 0.64648585, 0.71844033, 0.34313266],
    [0.09083225, 0.66152938, 0.06593091, 0.25857701, 0.96345285, 0.64026540],
    [0.11886697, 0.61505494, 0.90581639, 0.85530030, 0.41363143, 0.58523563],
    [0.63021764, 0.83809690, 0.68001305, 0.73189509, 0.52673671, 0.34842921],
    [0.76491917, 0.25588292, 0.60908422, 0.21807904, 0.32294277, 0.09579366],
    [0.05789554, 0.49167222, 0.24742222, 0.21811844, 0.42042833, 0.73096984],
    [0.19525188, 0.07922665, 0.55458046, 0.17056682, 0.01494418, 0.10703171],
    [0.64230298, 0.83687455, 0.02179269, 0.10148801, 0.68307083, 0.69241640],
    [0.78994255, 0.19554501, 0.57562333, 0.07365919, 0.25904917, 0.05109986],
    [0.52849733, 0.45742436, 0.36009569, 0.36204551, 0.81689098, 0.63747637],
    [0.72261522, 0.01181284, 0.06364591, 0.16517311, 0.07924415, 0.35995166],
    [0.07566492, 0.33450212, 0.13273274, 0.60831236, 0.91838592, 0.82233079],
    [0.94245084, 0.37743962, 0.48612233, 0.22879108, 0.08263175, 0.71195755],
    [0.14864702, 0.03394336, 0.72880565, 0.31606646, 0.02176938, 0.51691776],
    [0.81711239, 0.54816823, 0.10334758, 0.12436955, 0.72823482, 0.44967361],
    [0.41762629, 0.06409998, 0.24566877, 0.55904080, 0.19153138, 0.25464092],
    [0.72628566, 0.46489581, 0.92457051, 0.80724540, 0.63543840, 0.14341787],
    [0.31981043, 0.52009759, 0.29067775, 0.87670668, 0.49503469, 0.61908250],
    [0.87987128, 0.39796199, 0.00363456, 0.95699064, 0.26451373, 0.11486924],
    [0.54124078, 0.63140314, 0.03190205, 0.44998156, 0.79865282, 0.63370429],
    [0.22634792, 0.11502581, 0.82474966, 0.94538372, 0.90531153, 0.95101392],
    [0.68685257, 0.04101721, 0.00757301, 0.28500900, 0.69156848, 0.65554290],
    [0.17597754, 0.62441650, 0.29554198, 0.46955276, 0.09776977, 0.72814108],
    [0.88164674, 0.20445019, 0.41447436, 0.42038468, 0.26491501, 0.73066019],
    [0.06661051, 0.52804507, 0.81609520, 0.96101714, 0.08650933, 0.77778822],
    [0.93246638, 0.48881189, 0.25860774, 0.95624344, 0.19042781, 0.51985176],
    [0.84686697, 0.14242917, 0.06066859, 0.75629213, 0.55239830, 0.08130609],
    [0.80628208, 0.32412237, 0.72607601, 0.14871213, 0.71937640, 0.36288398],
    [0.47682313, 0.34094195, 0.01433523, 0.88013956, 0.99865470, 0.07966402],

    # Iteration results from our project
    [0.057895, 0.491672, 0.247422, 0.218118, 0.420428, 0.730969],
    [0.861674, 0.710584, 0.916207, 0.413415, 0.568112, 0.688380],
    [0.00754461, 0.05604467, 0.76901999, 0.05610508, 0.37627984, 0.80855666],
    [0.410355, 0.404626, 0.415739, 0.605124, 0.360664, 0.916164],
    [0.408786, 0.272011, 0.015960, 0.913927, 0.779885, 0.884180],
    [0.363794, 0.381721, 0.246385, 0.582342, 0.305840, 0.924433],
    [0.033969, 0.126740, 0.514117, 0.133496, 0.374088, 0.621625],
    [0.027231, 0.085405, 0.752894, 0.120046, 0.322614, 0.589848],
    [0.232502, 0.129287, 0.909833, 0.068397, 0.394889, 0.611060],
    [0.024767, 0.666722, 0.492560, 0.088079, 0.376655, 0.489081],
    [0.033972, 0.028757, 0.450864, 0.057456, 0.268097, 0.677730],
    [0.067142, 0.084164, 0.391241, 0.179196, 0.379401, 0.563393],
])

y = np.array([
    0.60443270,
    0.56275307,
    0.00750324,
    0.06142430,
    0.27304680,
    0.08374657,
    1.36496830,
    0.09264495,
    0.01786960,
    0.03356494,
    0.07351630,
    0.20630970,
    0.00882563,
    0.26840032,
    0.61152553,
    0.01479818,
    0.27489251,
    0.06676325,
    0.04211835,
    0.00270147,
    0.01820907,
    0.00701603,
    0.10050661,
    0.47539552,
    0.67514163,
    0.51645722,
    0.00377748,
    0.00313433,
    0.02134252,
    0.09541116,

    # Iteration outputs from our project
    1.36497012019043,
    0.0655047775737983,
    1.12760535430941,
    0.507529317551999,
    0.0007165677147890787,
    0.5763159658663746,
    2.402348648054967,
    1.7273032249712312,
    1.0330808540321927,
    0.7256545337861763,
    1.8200116920477367,
    2.3961650586254293,
])


# --------------------------------------------------
# 2. Helper functions
# --------------------------------------------------

def format_query(point, digits=6):
    """Format point for BBO portal submission."""
    return "-".join(f"{value:.{digits}f}" for value in point)


def clip_to_bounds(points):
    """Keep all candidate values inside [0, 1)."""
    return np.clip(points, 0.0, 0.999999)


def expected_improvement(candidates_scaled, model, scaler_y, y_best, xi=0.01):
    """Expected Improvement calculated on the original output scale."""
    mean_scaled, std_scaled = model.predict(candidates_scaled, return_std=True)

    mean = scaler_y.inverse_transform(mean_scaled.reshape(-1, 1)).ravel()
    std = std_scaled * scaler_y.scale_[0]
    std = np.maximum(std, 1e-12)

    improvement = mean - y_best - xi
    z = improvement / std

    ei = improvement * norm.cdf(z) + std * norm.pdf(z)
    return ei


def upper_confidence_bound(candidates_scaled, model, scaler_y, kappa=2.0):
    """Upper Confidence Bound calculated on the original output scale."""
    mean_scaled, std_scaled = model.predict(candidates_scaled, return_std=True)

    mean = scaler_y.inverse_transform(mean_scaled.reshape(-1, 1)).ravel()
    std = std_scaled * scaler_y.scale_[0]

    return mean + kappa * std


# --------------------------------------------------
# 3. Check data and best result
# --------------------------------------------------

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

best_index = np.argmax(y)
best_point = X[best_index]
best_y = y[best_index]

print("\nBest point so far:")
print(format_query(best_point))
print("Best output so far:", best_y)


# --------------------------------------------------
# 4. Scale data
# --------------------------------------------------

scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).ravel()

print("\nAfter scaling:")
print(f"X_scaled range: [{X_scaled.min():.3f}, {X_scaled.max():.3f}]")
print(f"y_scaled range: [{y_scaled.min():.3f}, {y_scaled.max():.3f}]")


# --------------------------------------------------
# 5. Gaussian Process model
# --------------------------------------------------
# Function 7 is high-dimensional, so we use scaling and controlled noise.

kernel = (
    ConstantKernel(1.0, constant_value_bounds=(1e-2, 1e3))
    * Matern(
        length_scale=np.ones(X.shape[1]),
        length_scale_bounds=(0.01, 1000.0),
        nu=2.5
    )
    + WhiteKernel(
        noise_level=1e-3,
        noise_level_bounds=(1e-6, 1e-1)
    )
)

gpr = GaussianProcessRegressor(
    kernel=kernel,
    alpha=0.01,
    normalize_y=False,
    n_restarts_optimizer=30,
    random_state=42
)

gpr.fit(X_scaled, y_scaled)

training_score = gpr.score(X_scaled, y_scaled)

print("\nGP Model Diagnostics:")
print("Kernel:", gpr.kernel_)
print("Training R²:", round(training_score, 4))

if training_score > 0.99:
    print("Warning: Training R² is very high. Check for possible overfitting.")


# --------------------------------------------------
# 6. Generate candidate points
# --------------------------------------------------
# Strategy:
# 40% local exploitation around the best point
# 60% global exploration because Function 7 is 6D and multimodal.

np.random.seed(42)

local_candidates = best_point + np.random.normal(
    loc=0.0,
    scale=0.12,
    size=(8000, X.shape[1])
)

local_candidates = clip_to_bounds(local_candidates)

sampler = qmc.LatinHypercube(d=X.shape[1], seed=42)
global_candidates = sampler.random(n=12000)

X_candidates = np.vstack([
    local_candidates,
    global_candidates
])

X_candidates_scaled = scaler_X.transform(X_candidates)


# --------------------------------------------------
# 7. Acquisition scoring: EI + UCB
# --------------------------------------------------

EI_XI = 0.01
UCB_KAPPA = 2.0

ei_scores = expected_improvement(
    candidates_scaled=X_candidates_scaled,
    model=gpr,
    scaler_y=scaler_y,
    y_best=best_y,
    xi=EI_XI
)

ucb_scores = upper_confidence_bound(
    candidates_scaled=X_candidates_scaled,
    model=gpr,
    scaler_y=scaler_y,
    kappa=UCB_KAPPA
)

ei_norm = (ei_scores - ei_scores.min()) / (ei_scores.max() - ei_scores.min() + 1e-12)
ucb_norm = (ucb_scores - ucb_scores.min()) / (ucb_scores.max() - ucb_scores.min() + 1e-12)

# Hybrid score:
# 55% EI for improvement
# 45% UCB for exploration
hybrid_score = 0.55 * ei_norm + 0.45 * ucb_norm


# --------------------------------------------------
# 8. Select next query
# --------------------------------------------------

best_candidate_index = np.argmax(hybrid_score)
x_next = X_candidates[best_candidate_index]

pred_scaled, std_scaled = gpr.predict(
    scaler_X.transform(x_next.reshape(1, -1)),
    return_std=True
)

predicted_y = scaler_y.inverse_transform(pred_scaled.reshape(-1, 1)).ravel()[0]
predicted_std = std_scaled[0] * scaler_y.scale_[0]

print("\nNext Point to Sample:")
print("X =", x_next)
print("Submission format:", format_query(x_next))
print("Predicted y:", predicted_y)
print("Uncertainty:", predicted_std)
print("Hybrid score:", hybrid_score[best_candidate_index])


# --------------------------------------------------
# 9. Show top 5 candidates
# --------------------------------------------------

top5_index = np.argsort(hybrid_score)[-5:][::-1]

print("\nTop 5 Candidates:")

for rank, idx in enumerate(top5_index, start=1):
    point = X_candidates[idx]

    pred_i_scaled, std_i_scaled = gpr.predict(
        scaler_X.transform(point.reshape(1, -1)),
        return_std=True
    )

    pred_i = scaler_y.inverse_transform(pred_i_scaled.reshape(-1, 1)).ravel()[0]
    std_i = std_i_scaled[0] * scaler_y.scale_[0]

    print(
        f"{rank}. X={format_query(point)} | "
        f"pred={pred_i:.6f}, "
        f"std={std_i:.6f}, "
        f"score={hybrid_score[idx]:.6f}"
    )


# --------------------------------------------------
# 10. Sanity checks
# --------------------------------------------------

print("\nSanity Checks:")
print(f"Training y range: [{y.min():.3f}, {y.max():.3f}]")
print(f"Best observed y: {best_y:.6f}")
print(f"Number of candidate points: {len(X_candidates)}")

Shape of X: (42, 6)
Shape of y: (42,)

Best point so far:
0.033969-0.126740-0.514117-0.133496-0.374088-0.621625
Best output so far: 2.402348648054967

After scaling:
X_scaled range: [-1.939, 2.254]
y_scaled range: [-0.750, 2.974]


/usr/local/lib/python3.12/dist-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-06. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(



GP Model Diagnostics:
Kernel: 0.73**2 * Matern(length_scale=[2.22, 1.87, 1.98, 2.06, 0.984, 1.35], nu=2.5) + WhiteKernel(noise_level=1e-06)
Training R²: 0.9997

Next Point to Sample:
X = [0.         0.13474717 0.40987577 0.10581726 0.3815078  0.61143612]
Submission format: 0.000000-0.134747-0.409876-0.105817-0.381508-0.611436
Predicted y: 2.434731867347576
Uncertainty: 0.08683681681983726
Hybrid score: 0.9931298071320545

Top 5 Candidates:
1. X=0.000000-0.134747-0.409876-0.105817-0.381508-0.611436 | pred=2.434732, std=0.086837, score=0.993130
2. X=0.000000-0.090541-0.425101-0.102321-0.400062-0.595806 | pred=2.424435, std=0.095234, score=0.963446
3. X=0.000000-0.107486-0.363521-0.129074-0.381379-0.578689 | pred=2.428385, std=0.089622, score=0.962129
4. X=0.000000-0.078417-0.382731-0.101863-0.380596-0.573247 | pred=2.413582, std=0.096018, score=0.898333
5. X=0.000000-0.166051-0.420894-0.103706-0.411201-0.596099 | pred=2.393981, std=0.112034, score=0.867668

Sanity Checks:
Training y ran